In [ ]:
import pandas as pd
from src.config import ALARM_K_CANDIDATES, FEATURE_SENSORS as F
from src.evaluate import evaluate, select_k, threshold
from src.features import prepare

p = prepare()


def distance_score(df: pd.DataFrame) -> pd.DataFrame:
    out = df[["unit", "cycle", "rul"]].copy()
    out["score"] = (df[F] ** 2).mean(axis=1)
    return out


thr = threshold(distance_score(p.val_healthy)["score"])
print("selected k:", select_k(distance_score(p.val), thr))
pd.DataFrame([evaluate(distance_score(p.val), thr, k) for k in ALARM_K_CANDIDATES])

selected k: 3


,k,threshold,units,early_alarm_rate,near_failure_units,detection_rate,lead_median,lead_q25,lead_q75,auroc_0_30,auroc_30_60,auroc_60_90
0,1,2.5128,20,0.10,20,1.0,74.5,49.25,100.25,1.0,0.928,0.726
1,2,2.5128,20,0.05,20,1.0,59.5,42.25,72.50,1.0,0.928,0.726
2,3,2.5128,20,0.00,20,1.0,58.0,40.75,65.50,1.0,0.928,0.726
3,5,2.5128,20,0.00,20,1.0,52.5,38.75,61.50,1.0,0.928,0.726


In [3]:
from src.baselines import SPE, T2
from src.run_baselines import scored

p = prepare()
X = p.fit_healthy[F].to_numpy()
rows = []
for n in [1, 2, 3, 4, 6, 8, 10, 12]:
    for name, cls in [("spe", SPE), ("t2", T2)]:
        m = cls(n_components=n).fit(X)
        thr = threshold(scored(p.val_healthy, m)["score"])
        val = scored(p.val, m)
        r = evaluate(val, thr, select_k(val, thr))
        rows.append(
            {
                "n": n,
                "stat": name,
                "det": r["detection_rate"],
                "lead": r["lead_median"],
                "auroc_30_60": r["auroc_30_60"],
                "auroc_60_90": r["auroc_60_90"],
            }
        )
pd.DataFrame(rows).pivot(index="n", columns="stat", values=["det", "lead", "auroc_30_60"])

det       lead       auroc_30_60       
stat   spe   t2   spe    t2         spe     t2
n                                             
1     0.75  0.8  65.0  31.5       0.847  0.687
2     0.50  1.0  20.5  72.0       0.579  0.962
3     0.50  1.0  16.5  67.0       0.580  0.948
4     0.45  1.0  17.0  56.0       0.572  0.940
6     0.45  1.0  17.0  59.0       0.576  0.928
8     0.45  1.0  11.0  51.5       0.543  0.911
10    0.45  1.0  16.0  50.5       0.561  0.900
12    0.40  1.0  11.0  51.5       0.536  0.891

In [2]:
import pandas as pd

from src.config import RESULTS

t = pd.read_csv(RESULTS / "test_results_FD001.csv", index_col=0)
t[["units", "near_failure_units", "auroc_0_30"]]

,units,near_failure_units,auroc_0_30
model,,,
distance,100.0,25.0,0.999
pca_spe,100.0,25.0,0.584
pca_t2,100.0,25.0,0.963
iforest,100.0,25.0,0.999
pca_t2_tuned,100.0,25.0,0.998
win_pca_spe,100.0,25.0,0.673
win_pca_t2,100.0,25.0,0.913
roll_distance,100.0,25.0,1.000
roll_pca_t2_tuned,100.0,25.0,0.997
